<a href="https://colab.research.google.com/github/0szgn0/Titanic-Survival-Prediction/blob/main/Titanic%20by%20szgn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

titanic_path = kagglehub.competition_download('titanic')

print('Data source import complete.')


In [ ]:
import kagglehub
import pandas as pd

import kagglehub
kagglehub.login()
path = kagglehub.competition_download('titanic')

In [ ]:
df_train = pd.read_csv(f"{path}/train.csv")
df_test = pd.read_csv(f"{path}/test.csv")
df_submission = pd.read_csv(f"{path}/gender_submission.csv")

df_train.head()
df_train.info()
display(df_train.describe())

In [ ]:
# Handling Missing Values

# Fill missing 'Age' based on the median of their Pclass and Sex group
df_train['Age'] = df_train.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

# Fill missing 'Embarked' with the most frequent value ('S')
df_train['Embarked'] = df_train['Embarked'].fillna(df_train['Embarked'].mode()[0])

# Drop the 'Cabin' column due to too many missing values, and 'Ticket' as it requires complex parsing
df_train = df_train.drop(columns=['Cabin', 'Ticket'])


# Feature Engineering

# Create a new feature for total family size
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch'] + 1

# Check our work
print("Missing values after cleaning:")
print(df_train.isnull().sum())
display(df_train.head())

In [ ]:
df_train['Sex'] = df_train['Sex'].map({'male': 0, 'female': 1})

df_train = pd.get_dummies(df_train, columns=['Embarked'], drop_first=True)

df_train = df_train.drop(columns=['PassengerId', 'Name'])

print(df_train.info())
display(df_train.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Feature and Target Definition
X = df_train.drop(columns=['Survived'])
y = df_train['Survived']

# Train-Test Split (80% Train, 20% Validation)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Initialization
rf_model = RandomForestClassifier(random_state=42)

# Model Training
rf_model.fit(X_train, y_train)

# Prediction
y_pred = rf_model.predict(X_test)

# Accuracy Evaluation
accuracy = accuracy_score(y_test, y_pred)

print(f"Validation Accuracy: % {accuracy * 100:.2f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameter Grid Definition
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Base Model Initialization
rf_base = RandomForestClassifier(random_state=42)

# Grid Search Setup (5-fold Cross Validation)
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Grid Search Execution
grid_search.fit(X_train, y_train)

# Best Parameters and Score Extraction
best_rf_model = grid_search.best_estimator_
optimized_accuracy = best_rf_model.score(X_test, y_test)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Optimized Validation Accuracy: % {optimized_accuracy * 100:.2f}")

In [ ]:
# Test Data Preprocessing (Missing Values)
df_test['Age'] = df_test.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
df_test['Fare'] = df_test['Fare'].fillna(df_test['Fare'].median())

# Feature Engineering
df_test['FamilySize'] = df_test['SibSp'] + df_test['Parch'] + 1

# Categorical Encoding
df_test['Sex'] = df_test['Sex'].map({'male': 0, 'female': 1})
df_test = pd.get_dummies(df_test, columns=['Embarked'], drop_first=True)

# Feature Selection
X_test_kaggle = df_test.drop(columns=['PassengerId', 'Name', 'Cabin', 'Ticket'])

# Optimized Model Initialization
from sklearn.ensemble import RandomForestClassifier
rf_optimized_final = RandomForestClassifier(
    max_depth=5,
    min_samples_leaf=2,
    min_samples_split=10,
    n_estimators=300,
    random_state=42
)

# Full Dataset Training
rf_optimized_final.fit(X, y)

# Final Prediction
predictions = rf_optimized_final.predict(X_test_kaggle)

In [ ]:
submission = pd.DataFrame({f"PassengerId": df_test["PassengerId"],
"Survived": predictions
})
submission.to_csv('submission.csv', index=False)